# 215 — Concatenated clusters: cross-correlation and lead/lag

`214` asks **when** each cluster comes on. This asks a different question:

> **do the clusters lead and lag one another, and by how much?**

One cluster is fixed as the **reference** — the auditory cluster — and every other
cluster's mean HG trace is correlated against it across a range of time shifts. The
output is the full **r vs lag** curve, not a single summary number.

### How to read the lag

    r(tau) = Pearson[ ref(t), other(t + tau) ]

    tau > 0   the other cluster happens LATER    (the reference leads it)
    tau < 0   the other cluster happens EARLIER  (the reference lags it)
    tau = 0   they rise and fall together

Verified on synthetic data in both directions. Lags are in **warped-time %**, so they
are an **ordering**, not a latency in milliseconds.

### Read this before trusting any number here

Cross-correlation only means something when two signals are **the same shape, shifted in
time**. That has to be checked, not assumed — and here it fails outright in one window:

| window | lags agreeing with the actual peak-time difference |
|---|---|
| full trial (0–100%) | **0 of 8** |
| response only (50–100%) | **8 of 8** |

Over the full trial these cluster means are *not* shifted copies: one has an early
sensory transient, another only a late production bump, so no shift maps one onto the
other and the "peak lag" is whatever maximises a weak correlation. Inside the response
window every cluster has a production-period bump, the shapes genuinely are shifted
versions, and the lags line up with real timing differences.

**`window='response'` is the analysis. `window='full'` is diagnostic only** — section 4
runs it specifically to show the failure rather than hide it.

## 1. Setup and the reference cluster

Everything adjustable is in one cell. The reference is set **manually** on purpose:
which cluster counts as "the auditory one" is a judgement about the data, not something
to be inferred silently — and cluster ids change whenever the clustering is re-fit.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path('..').resolve()))
sys.path.insert(0, str(Path('.').resolve()))

from functions import lf_cluster_timing as T

SCRIPT_NAME = '215_concat_crosscorrelation.ipynb'
pd.set_option('display.width', 170)

In [ ]:
# ────────────────────────  ADJUST THESE  ────────────────────────
XC_RUN     = 'outputs/clustering/kmeans/concat_hg/runs/20260803_175417'
XC_K       = 10          # which K in the sweep
# REF_CLUSTER is NOT set here. Cluster ids are arbitrary and change on every
# re-fit, and this line used to say `= 1` while every published number had been
# computed against c4 — the notebook could not reproduce its own figures. It is
# now chosen by PROFILE, below, which is what the write-up always claimed.
XC_WINDOW  = 'response'  # 'response' (50-100%, the valid one) | 'full' | 'stimulus'

MIN_OVERLAP = 0.60       # sets the lag range: +/- (1-MIN_OVERLAP) x window length.
                         # 0.8 censored half the peaks at the search edge. 0.6 does NOT
                         # censor none: 8 of 27 pairs have a true offset outside +/-20%.
SHAPE_GATE  = 0.30       # |peak r| below this -> shapes differ, the lag is not a delay
N_BOOT      = 400        # bootstrap over electrodes, for the CI on the lag
N_SHUFFLE   = 200        # label-permutation null on the lag
# ─────────────────────────────────────────────────────────────

XC_RUN_DIR = Path(XC_RUN)
X_xc  = np.load(XC_RUN_DIR / 'X_train.npy')
lab_xc = T.load_labels_by_k(XC_RUN_DIR)[XC_K]

import collections
sizes = collections.Counter(lab_xc.tolist())
print(f'run  : {XC_RUN_DIR.name}   n={len(lab_xc)}   K={XC_K}')
print(f'sizes: {dict(sorted(sizes.items()))}')

# The reference is picked from the data: the ACTIVATION with the earliest
# stimulus-window onset under audio and no stimulus onset in the other two
# conditions. On this run that is c4 (n=41).
_t = T.cluster_timing(X_xc, lab_xc)
REF_CLUSTER = T.pick_reference_cluster(_t)
print(f'ref  : c{REF_CLUSTER}  (n={sizes[REF_CLUSTER]})  <- chosen by profile, not by id')
print()
print(f'--- c{REF_CLUSTER} profile (is it auditory?) ---')
print(_t[_t.cluster == REF_CLUSTER][
    ['condition', 'n', 'onset_stim_pct', 'onset_resp_pct',
     'peak_stim_db', 'peak_resp_db']].round(2).to_string(index=False))
print('\nExpect: an early stimulus onset with a large positive peak in AUDIO,')
print('and no stimulus onset in picture / reading.')

## 2. The lag profiles

One panel per condition, one curve per cluster, the peak marked.

Curves that fail the interpretability gate are drawn **dashed and pale**. They are shown
rather than dropped, because "these two shapes do not align at any lag" is itself a
result — but they must not be read as a measured delay. A cluster fails the gate when
its peak `r` is below `SHAPE_GATE`, when the peak sits at the edge of the searched lag
range (censored, not measured), or when the bootstrap CI is wider than 20% of the trial.

In [ ]:
xc_tab, xc_prof = T.cluster_xcorr(
    X_xc, lab_xc, ref_cluster=REF_CLUSTER, window=XC_WINDOW,
    min_overlap=MIN_OVERLAP, shape_gate=SHAPE_GATE,
    n_boot=N_BOOT, n_shuffle=N_SHUFFLE)

# colour clusters by onset order so they match FIG 2.11 / section 3
xc_order = T.onset_order(_t[_t.cluster != REF_CLUSTER])

T.plot_xcorr_profiles(xc_tab, xc_prof, ref_cluster=REF_CLUSTER, order=xc_order,
                      shape_gate=SHAPE_GATE,
                      title=f'Cross-correlation vs c{REF_CLUSTER} · K={XC_K} · {XC_WINDOW} window')
plt.show()

cols = ['condition', 'cluster', 'n', 'peak_lag_pct', 'peak_r',
        'ci_lo_pct', 'ci_hi_pct', 'at_edge', 'p_shuffle', 'interpretable', 'reason']
print(xc_tab[cols].round(2).to_string(index=False))
print()
print(f"interpretable: {int(xc_tab['interpretable'].sum())} / {len(xc_tab)}")
print('failures:', {k: v for k, v in xc_tab.loc[~xc_tab.interpretable, 'reason']
                    .value_counts().items()})

### What each column means, and what it does not

* **`peak_lag_pct`** — the shift, in % of the warped trial, at which the two clusters
  best align. Positive = that cluster lags behind the reference. This is warped time,
  so it is an ordering, **not a latency in ms**.
* **`peak_r`** — how well the shapes align *at all* at that shift. This is the number
  that decides whether the lag means anything. A lag with `r = 0.95` is a shift; a lag
  with `r = 0.25` is two different shapes and a meaningless argmax.
* **`ci_lo/hi`** — bootstrap over the electrodes *within* each cluster. This is the
  precision of the lag and the most honest column in the table: a lag of −11% with CI
  [−14, −8] is a result, the same −11% with CI [−28, +5] is not.
* **`p_shuffle`** — label-permutation null **on the lag**. Treat it as weak evidence:
  it only rules out "the lag is exactly zero".

  Why it is weak, and why the obvious alternative is worse: shuffling cluster labels
  makes both groups random samples of the same population, so both means collapse
  toward the grand mean and the null lag is ~0. Almost any real lag beats that. Running
  the same shuffle against **peak r** instead is not weak but actively useless — the
  shuffled means correlate at ~0.97, giving **p = 1.000** for every pair. The bootstrap
  CI, not the p-value, is what should be reported.

## 3. Save

Written next to the run's other timing output, so the figure and the table travel with the clustering they describe.

In [ ]:
# Save
XC_DIR = XC_RUN_DIR / 'timing'
XC_DIR.mkdir(parents=True, exist_ok=True)
stem = f'xcorr_K{XC_K}_ref-c{REF_CLUSTER}_{XC_WINDOW}'
xc_tab.to_csv(XC_DIR / f'{stem}.csv', index=False)
T.plot_xcorr_profiles(xc_tab, xc_prof, ref_cluster=REF_CLUSTER, order=xc_order,
                      shape_gate=SHAPE_GATE,
                      title=f'Cross-correlation vs c{REF_CLUSTER} · K={XC_K} · {XC_WINDOW} window',
                      out_png=XC_DIR / f'{stem}.png')
print('wrote', XC_DIR / f'{stem}.csv')
print('wrote', XC_DIR / f'{stem}.png')

## 4. Why the full-trial window is not used

Not a formality — this is the check that decides whether section 8 is worth anything.

For each cluster the measured lag is compared against the **difference in peak times**,
which is what a lag should approximate when two signals really are shifted copies. If
the two disagree, the cross-correlation is not measuring a shift.

Expect the full trial to fail on essentially every cluster and the response window to
pass on essentially all of them. If that ever flips, section 8's window should change
with it.

In [ ]:
rows = []
for win in ('full', 'response'):
    tb, _ = T.cluster_xcorr(X_xc, lab_xc, ref_cluster=REF_CLUSTER, window=win,
                            min_overlap=MIN_OVERLAP, shape_gate=SHAPE_GATE,
                            n_boot=0 if False else 50, n_shuffle=50)
    Xb = T.block_view(X_xc, 3); nt = Xb.shape[2]
    sl = T._window_slice(nt, win)
    for _, r in tb[tb.condition == 'audio'].iterrows():
        a = Xb[lab_xc == REF_CLUSTER, 0, sl].mean(0)
        b = Xb[lab_xc == int(r.cluster), 0, sl].mean(0)
        diff = (int(np.argmax(b)) - int(np.argmax(a))) / nt * 100
        agree = (np.sign(diff) == np.sign(r.peak_lag_pct)) or abs(diff) < 3
        rows.append(dict(window=win, cluster=int(r.cluster),
                         peak_time_diff_pct=round(diff, 1),
                         measured_lag_pct=round(r.peak_lag_pct, 1),
                         peak_r=round(r.peak_r, 2),
                         sign_agrees=bool(agree and not r.at_edge)))
chk = pd.DataFrame(rows)
print(chk.to_string(index=False))
print()
for win in ('full', 'response'):
    s = chk[chk.window == win]
    print(f"  {win:9s}: {int(s.sign_agrees.sum())}/{len(s)} lags agree with the "
          f"peak-time difference")